In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import builtins
import numpy as np
import torch

import lazyslide as zs
import lazyslide.models.vision.gigapath as gp
from wsidata import open_wsi

# -------------------------------------------------------------------
# This is a hack to fix an issue with the GigaPathSlideEncoder where np
# is not loaded correctly in the namespace
builtins.np = np

# This patch fixes inconsistencies in GigaPathSlideEncoder.encode_slide


def patched_encode_slide(self, tile_embed, coordinates):
    out = self.model(tile_embed, coordinates)
    if isinstance(out, list):
        out = out[0]
    return out.squeeze()


gp.GigaPathSlideEncoder.encode_slide = patched_encode_slide
# -------------------------------------------------------------------


def slide_embedding(
    image: str,
    target_dir: str | Path | None = None,
    batch_size: int = 256,
    num_workers: int = 16,
    tile_px: int = 256,
    mpp: float = 0.5,
    **kwargs: Any
) -> None:
    """
    Extract slide-level embedding using GigaPath encoder and save to disk.

    Tile-level features are saved as .h5ad, slide-level aggregated
    embedding as .pt, both in the target directory.
    """
    wsi = open_wsi(image, attach_thumbnail=False)

    zs.pp.find_tissues(wsi)
    zs.pp.tile_tissues(wsi, tile_px, mpp=mpp, **kwargs)

    zs.tl.feature_extraction(
        wsi,
        "gigapath",
        batch_size=batch_size,
        num_workers=num_workers,
        amp=True,
        pbar=True,
    )

    zs.tl.feature_aggregation(
        wsi,
        "gigapath",
        encoder="gigapath-slide-encoder",
        amp=True,
        device="cuda",
    )

    # Tile-level: (n_tiles, 1536)
    adata = wsi.tables["gigapath_tiles"]

    # Slide-level aggregated: (768, )
    # compress to 768 dimensions from 1536 dimensions per tile
    slide_embeddings = torch.as_tensor(
        adata.uns["agg_ops"]["agg_slide"]["features"],
        dtype=torch.float32,
    ).squeeze().cpu()

    target_dir = Path(target_dir) if target_dir else Path(image).parent

    stem = Path(image).stem
    target_h5ad = target_dir / f"{stem}.h5ad"
    target_pt = target_dir / f"{stem}.pt"

    target_h5ad.parent.mkdir(parents=True, exist_ok=True)

    adata.write_h5ad(target_h5ad)

    torch.save(slide_embeddings, target_pt)
    print(f"Saved tile-level features to {target_h5ad}")
    print(f"Saved slide-level features to {target_pt}")